In [ ]:
%run ./_bootstrap

In [ ]:
from pyspark.sql import SparkSession
from helpers.classifier_model import build_logreg_router
from helpers.classifier_eval import train_eval_classifier_doc_split
from helpers.classifier_mlflow import log_classifier_run
from helpers.registry import append_classifier_run_id
from helpers.classifier_data import build_classifier_training_df

In [ ]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [ ]:
EXP_NAME = "/Shared/medibot_classifier"
allowed_labels = ["drug", "condition"]

In [ ]:
pdf = build_classifier_training_df(
    spark,
    source_table="workspace.med.doc_chunks",
    allowed_labels=allowed_labels,
    chunk_text_max_chars=1200,
    min_rows_per_class=200,
)

X = pdf["train_text"].astype(str).tolist()
y = pdf["label"].astype(str).tolist()

clf = build_logreg_router(
    ngram_range=(1, 2),
    word_max_features=100000,
    min_df=2,
    max_df=0.9,
    logreg_c=2.0,
    max_iter=2000,
)

clf, metrics = train_eval_classifier_doc_split(
    clf,
    pdf,
    test_size=0.15,
    random_seed=42,
)

print("Train docs:", metrics["train_docs"], "Test docs:", metrics["test_docs"])
print(metrics["report_text"])
print("Confusion matrix:", metrics["confusion_matrix"])

print(metrics["report_text"])
print("Confusion matrix:", metrics["confusion_matrix"])

params = {
    "source_table": "workspace.med.doc_chunks",
    "allowed_labels": ",".join(allowed_labels),
    "chunk_text_max_chars": 1200,
    "tfidf_ngram_range": "(1,2)",
    "tfidf_stop_words": "english",
    "tfidf_min_df": 2,
    "tfidf_max_df": 0.9,
    "tfidf_max_features": 100000,
    "logreg_solver": "lbfgs",
    "logreg_C": 2.0,
    "logreg_max_iter": 2000,
    "test_size": 0.15,
    "random_seed": 42,
}

logged = log_classifier_run(
    clf=clf,
    params=params,
    metrics=metrics,
    exp_name=EXP_NAME,
    run_name="question_router_logreg",
    artifact_path="model",
)

print("RUN_ID:", logged["run_id"])
print("MODEL_URI:", logged["model_uri"])

append_classifier_run_id(spark, logged["run_id"])
print("Wrote to classifier registry.")